# Terra Softech Week 1 - E-Commerce Sales Data Preprocessing

- **Module A:** Data loading, schema assessment, missing-value review, unique-count statistics, and raw baseline metrics.
- **Module B:** Data type standardization, anomaly removal, duplicate handling, feature engineering, data quality log, and cleaned CSV export.

## 1. Environment


In [1]:
# 这里先把本周会用到的包统一导入。
# pandas 是主力，用来读表、查缺失、清洗和导出；numpy 主要用来做数值校验。
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 显示设置：防止表格列太多时被折叠得看不清。
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 图表风格先设好，后面如果做 EDA 可视化可以直接沿用。
sns.set_theme(style="whitegrid")

## 2. Module A - Data Loading


In [2]:
NOTEBOOK_DIR = Path.cwd()

# 这里优先使用提交包里的正式文件名 origin_data.csv。
# 后面的 data.csv 是 fallback：如果别人用旧文件名，也还能跑。
candidate_data_paths = [
    NOTEBOOK_DIR / "dataset" / "origin_data.csv",
    NOTEBOOK_DIR / "dataset" / "data.csv",
    NOTEBOOK_DIR.parent / "dataset" / "origin_data.csv",
    NOTEBOOK_DIR.parent / "dataset" / "data.csv",
    Path("dataset") / "origin_data.csv",
    Path("dataset") / "data.csv",
]

DATA_PATH = next((path for path in candidate_data_paths if path.exists()), None)
if DATA_PATH is None:
    searched_paths = "\n".join(str(path) for path in candidate_data_paths)
    raise FileNotFoundError(f"Raw dataset was not found. Searched paths:\n{searched_paths}")

# 按 Week 1 任务要求，清洗后的正式交付文件名是 cleaned_sales.csv。
DATASET_DIR = DATA_PATH.parent
OUTPUT_PATH = DATASET_DIR / "cleaned_sales.csv"

# 这份数据里有 £ 这类特殊字符，默认 utf-8 会读失败，所以这里用 latin1。
# CustomerID 是编号，不是用来计算的数字；读入时直接设成 string，避免后面 float -> int -> string 的绕路转换。
df_raw = pd.read_csv(
    DATA_PATH,
    encoding="latin1",
    dtype={"CustomerID": "string"},
)

# 先看数据规模和前几行，确认读进来的确实是交易明细表。
print(f"Loaded raw dataset from: {DATA_PATH}")
print(f"Cleaned dataset will be saved to: {OUTPUT_PATH}")
print(f"Raw dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns")

df_raw.head()


Loaded raw dataset from: /Users/jia/Desktop/工作/Terra Softech/Code/week1总/Week1_0706/dataset/origin_data.csv
Cleaned dataset will be saved to: /Users/jia/Desktop/工作/Terra Softech/Code/week1总/Week1_0706/dataset/cleaned_sales.csv
Raw dataset shape: 541,909 rows x 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


## 3. Data Overview


In [3]:
# 摸底：每一列是什么类型、缺不缺值、缺多少。
# 这一步的目的先知道原始表的问题在哪里。
schema_table = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [str(dtype) for dtype in df_raw.dtypes],
    "non_null_count": df_raw.notna().sum().values,
    "null_count": df_raw.isna().sum().values,
    "null_ratio": (df_raw.isna().mean().values * 100).round(2),
})

print("Raw DataFrame info():")
buffer = StringIO()
df_raw.info(buf=buffer)
print(buffer.getvalue())

schema_table


Raw DataFrame info():
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  string 
 7   Country      541909 non-null  str    
dtypes: float64(1), int64(1), str(5), string(1)
memory usage: 33.1 MB



,column,dtype,non_null_count,null_count,null_ratio
0,InvoiceNo,str,541909,0,0.00
1,StockCode,str,541909,0,0.00
2,Description,str,540455,1454,0.27
3,Quantity,int64,541909,0,0.00
4,InvoiceDate,str,541909,0,0.00
5,UnitPrice,float64,541909,0,0.00
6,CustomerID,string,406829,135080,24.93
7,Country,str,541909,0,0.00


## 4. Missing Values and Unique Counts

In [4]:
# 这里把缺失值和唯一值放到一张表里看。
# missing_count 看脏数据程度，unique_count 看这个字段大概是什么粒度。
missing_summary = (
    pd.DataFrame({
        "missing_count": df_raw.isna().sum(),
        "missing_ratio_pct": (df_raw.isna().mean() * 100).round(2),
        "unique_count": df_raw.nunique(dropna=True),
    })
    .reset_index()
    .rename(columns={"index": "column"})
)

key_fields = [
    "InvoiceNo",
    "CustomerID",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "InvoiceDate",
    "Country",
]

missing_summary[missing_summary["column"].isin(key_fields)]


,column,missing_count,missing_ratio_pct,unique_count
0,InvoiceNo,0,0.00,25900
1,StockCode,0,0.00,4070
2,Description,1454,0.27,4223
3,Quantity,0,0.00,722
4,InvoiceDate,0,0.00,23260
5,UnitPrice,0,0.00,1630
6,CustomerID,135080,24.93,4372
7,Country,0,0.00,38


## 5. Raw Baseline Metrics


In [5]:
# 这里先算清洗前的销售额
# 因为原始数据里还有退货、0 单价、缺失客户、重复行等问题。
raw_revenue = (df_raw["Quantity"] * df_raw["UnitPrice"]).sum()

# 用一张小表记录原始规模：多少交易行、多少订单、多少客户、多少商品、粗略销售额。
raw_metrics = pd.DataFrame([
    {"metric": "Total raw order-line records", "value": f"{len(df_raw):,}"},
    {"metric": "Unique invoices / orders", "value": f"{df_raw['InvoiceNo'].nunique(dropna=True):,}"},
    {"metric": "Unique customers", "value": f"{df_raw['CustomerID'].nunique(dropna=True):,}"},
    {"metric": "Unique products", "value": f"{df_raw['StockCode'].nunique(dropna=True):,}"},
    {"metric": "Raw revenue before cleaning", "value": f"{raw_revenue:,.2f}"},
])

raw_metrics


,metric,value
0,Total raw order-line records,"541,909"
1,Unique invoices / orders,"25,900"
2,Unique customers,"4,372"
3,Unique products,"4,070"
4,Raw revenue before cleaning,"9,747,747.93"


## 6. Dataset Field Mapping


| Task Field | Actual CSV Field | Notes |
|---|---|---|
| Order ID | `InvoiceNo` | 发票/订单编号。一个订单可以有多个商品行，所以不能只按它去重。 |
| Product ID | `StockCode` | 商品编号。 |
| Customer ID | `CustomerID` | 客户编号。后面做客户分析和 LTV 必须用它。 |
| Order Date | `InvoiceDate` -> `OrderDate` | `InvoiceDate` 是原始字符串，`OrderDate` 是标准化后的日期字段。最终 cleaned 表只保留 `OrderDate`。 |
| Region | `Country` | 当前数据里能用的地区字段。 |
| Quantity | `Quantity` | 商品数量。 |
| Unit Price | `UnitPrice` | 商品单价。 |
| Category | Not available | 原始数据没有，不主动虚构。 |
| Payment Method | Not available | 原始数据没有，不主动虚构。 |


## 7. Module B - Cleaning and Preparation

Create a reproducible cleaning pipeline:

1. Standardize column types.
2. Trim string fields.
3. Remove invalid numeric records.
4. Remove rows without `CustomerID`.
5. Drop exact duplicate rows.
6. Add `Revenue` and `OrderMonth`.


In [6]:
# 这里不再每一步都复制一整张大表。
# 思路是：先统一字段格式，再用一个 mask 一次性筛掉无效记录，减少大表重复复制。
raw_row_count = len(df_raw)
raw_missing_ratios = df_raw[["Quantity", "UnitPrice", "CustomerID", "InvoiceDate"]].isna().mean()

# 字符串字段先统一去掉前后空格。
# assign 会返回一张处理后的表，比多次 copy 更适合这种连续清洗流程。
df_clean = df_raw.assign(
    InvoiceNo=df_raw["InvoiceNo"].astype("string").str.strip(),
    StockCode=df_raw["StockCode"].astype("string").str.strip(),
    Description=df_raw["Description"].astype("string").str.strip(),
    Country=df_raw["Country"].astype("string").str.strip(),
    CustomerID=df_raw["CustomerID"].str.strip().replace("", pd.NA),
    OrderDate=pd.to_datetime(df_raw["InvoiceDate"], errors="coerce"),
    Quantity=pd.to_numeric(df_raw["Quantity"], errors="coerce"),
    UnitPrice=pd.to_numeric(df_raw["UnitPrice"], errors="coerce"),
)

# 过滤前先把问题数量记下来，方便解释到底删了哪些数据。
missing_customer_rows = int(df_clean["CustomerID"].isna().sum())
missing_order_date_rows = int(df_clean["OrderDate"].isna().sum())
non_positive_numeric_rows = int(((df_clean["Quantity"] <= 0) | (df_clean["UnitPrice"] <= 0)).sum())

# 用一个布尔 mask 表示“正常销售记录”。
# 这样可以把缺失值过滤和非正数过滤合在一起做，避免每过滤一步就复制一张大表。
valid_sales_mask = (
    df_clean[["Quantity", "UnitPrice", "CustomerID", "OrderDate"]].notna().all(axis=1)
    & (df_clean["Quantity"] > 0)
    & (df_clean["UnitPrice"] > 0)
)

# duplicate_rows_before_drop 统计的是通过核心过滤后，仍然完全重复的交易行数量。
duplicate_rows_before_drop = int(df_clean.loc[valid_sales_mask].duplicated().sum())

# 这里用链式写法完成最终清洗：筛选正常销售记录 -> 去重 -> 统一类型 -> 新增分析字段 -> 删除原始日期字符串。
df_clean = (
    df_clean.loc[valid_sales_mask]
    .drop_duplicates()
    .assign(
        Quantity=lambda x: x["Quantity"].astype("int64"),
        UnitPrice=lambda x: x["UnitPrice"].astype("float64"),
        CustomerID=lambda x: x["CustomerID"].astype("string"),
        Revenue=lambda x: x["Quantity"] * x["UnitPrice"],
        OrderMonth=lambda x: x["OrderDate"].dt.to_period("M").astype(str),
    )
    .drop(columns=["InvoiceDate"])
)

# 看一下清洗前后行数变化。
final_row_count = len(df_clean)

print(f"Raw rows: {raw_row_count:,}")
print(f"Rows after cleaning: {final_row_count:,}")
print(f"Rows removed: {raw_row_count - final_row_count:,}")

df_clean.head()


Raw rows: 541,909
Rows after cleaning: 392,692
Rows removed: 149,217


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,OrderDate,Revenue,OrderMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,17850,United Kingdom,2010-12-01 08:26:00,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,17850,United Kingdom,2010-12-01 08:26:00,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12


## 8. Data Quality Log

In [7]:
# 这里继续做清洗后缺失率检查。
# InvoiceDate 已经删除，所以清洗后用 OrderDate 代表标准化后的订单时间。
clean_missing_ratios = df_clean[["Quantity", "UnitPrice", "CustomerID", "OrderDate"]].isna().mean()

# Data Quality Log 用来回答 manager 常问的问题：原始多少行，删了什么，最后剩多少。
quality_log = pd.DataFrame([
    {"check": "Raw row count", "value": raw_row_count},
    {"check": "Rows with missing CustomerID before filtering", "value": missing_customer_rows},
    {"check": "Rows with invalid/missing OrderDate before filtering", "value": missing_order_date_rows},
    {"check": "Rows with non-positive Quantity or UnitPrice before filtering", "value": non_positive_numeric_rows},
    {"check": "Exact duplicate rows removed after core filters", "value": duplicate_rows_before_drop},
    {"check": "Final cleaned row count", "value": final_row_count},
    {"check": "Total rows removed", "value": raw_row_count - final_row_count},
])

# 缺失率 before / after 对比。
# raw 里字段叫 InvoiceDate，cleaned 里对应的是 OrderDate，所以这里统一展示成 OrderDate。
null_ratio_log = pd.DataFrame({
    "critical_field": ["Quantity", "UnitPrice", "CustomerID", "OrderDate"],
    "raw_null_ratio_pct": (raw_missing_ratios.values * 100).round(2),
    "cleaned_null_ratio_pct": (clean_missing_ratios.values * 100).round(2),
})

print("Data quality log:")
display(quality_log)

print("Critical-field null ratio before vs after cleaning:")
null_ratio_log


Data quality log:


,check,value
0,Raw row count,541909
1,Rows with missing CustomerID before filtering,135080
2,Rows with invalid/missing OrderDate before fil...,0
3,Rows with non-positive Quantity or UnitPrice b...,11805
4,Exact duplicate rows removed after core filters,5192
5,Final cleaned row count,392692
6,Total rows removed,149217


Critical-field null ratio before vs after cleaning:


,critical_field,raw_null_ratio_pct,cleaned_null_ratio_pct
0,Quantity,0.00,0.00
1,UnitPrice,0.00,0.00
2,CustomerID,24.93,0.00
3,OrderDate,0.00,0.00


## 9. Validation Checks


In [8]:
# 这里用 assert 做最后检查。
# assert 可以理解成自动验收：条件不满足就直接报错，避免把脏数据导出去。

# OrderDate 必须是真正的日期类型，后面才能提取月份、星期、小时等时间特征。
assert pd.api.types.is_datetime64_any_dtype(df_clean["OrderDate"]), "OrderDate must be datetime."

# 清洗后只保留正常销售记录，所以数量和单价都必须大于 0。
assert (df_clean["Quantity"] > 0).all(), "Quantity must be positive after cleaning."
assert (df_clean["UnitPrice"] > 0).all(), "UnitPrice must be positive after cleaning."

# CustomerID 不能为空，否则后面客户特征、复购、LTV 都做不了。
assert df_clean["CustomerID"].notna().all(), "CustomerID must not contain null values after cleaning."

# Revenue 必须等于 Quantity * UnitPrice。
# 用 allclose 是为了避免浮点数小数位造成误判。
assert np.allclose(df_clean["Revenue"], df_clean["Quantity"] * df_clean["UnitPrice"]), "Revenue formula mismatch."

# OrderMonth 是从 OrderDate 得到的，不能有缺失。
assert df_clean["OrderMonth"].notna().all(), "OrderMonth must be populated."

# 清洗后不应该还有完全重复行。
assert df_clean.duplicated().sum() == 0, "Cleaned dataset still contains exact duplicate rows."

# 最终 cleaned 表只保留标准日期字段，不再保留原始 InvoiceDate 字符串。
assert "InvoiceDate" not in df_clean.columns, "InvoiceDate should be removed from cleaned dataset."

print("All validation checks passed.")


All validation checks passed.


## 10. Export Cleaned Dataset


In [9]:
# 把清洗后的表导出成 CSV。
# index=False 是为了不把 pandas 自动生成的行号写进文件。
df_clean.to_csv(OUTPUT_PATH, index=False)

# 导出后再读回来一次，确认这个文件后续可以被新的 notebook 直接使用。
reloaded = pd.read_csv(OUTPUT_PATH)

print(f"Cleaned dataset exported to: {OUTPUT_PATH}")
print(f"Reloaded cleaned dataset shape: {reloaded.shape[0]:,} rows x {reloaded.shape[1]:,} columns")

reloaded.head()


Cleaned dataset exported to: /Users/jia/Desktop/工作/Terra Softech/Code/week1总/Week1_0706/dataset/cleaned_sales.csv
Reloaded cleaned dataset shape: 392,692 rows x 10 columns


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,OrderDate,Revenue,OrderMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,17850,United Kingdom,2010-12-01 08:26:00,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,17850,United Kingdom,2010-12-01 08:26:00,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,17850,United Kingdom,2010-12-01 08:26:00,20.34,2010-12


## 11. Week 1 Summary

The notebook now provides a reproducible Week 1 foundation: raw data overview, quality checks, standardized columns, anomaly filtering, duplicate removal, engineered revenue/month fields, and an exported `cleaned_sales.csv` file for next week's analysis tasks.